## 1. Detect zipped or automatically unzipped input


In [ ]:
from pathlib import Path
import base64, json, os, subprocess, sys, time, zipfile
INPUT_ROOT = Path(os.environ.get("JCAM_INPUT_ROOT", "/kaggle/input"))
WORK_ROOT = Path(os.environ.get("JCAM_WORK_ROOT", "/kaggle/working"))
RUN_ROOT = WORK_ROOT / "jcam_thm10_fixed_k2"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
RUN_OK = True
STAGES = []
def record(stage, outcome, detail):
    STAGES.append({"stage": stage, "outcome": outcome, "detail": detail})
    print(stage, outcome, detail)
mode = "embedded_exact_sources"
if INPUT_ROOT.exists():
    if any(INPUT_ROOT.rglob("check_thm10_fixed_k2.py")):
        mode = "kaggle_auto_unzipped_detected; embedded exact sources selected"
    elif any(INPUT_ROOT.rglob("*.zip")):
        mode = "kaggle_zip_detected; embedded exact sources selected"
record("bootstrap", "passed", mode)


## 2. Install exact source package from notebook cell


In [ ]:
SOURCE_ARCHIVE = base64.b64decode("".join([
    'UEsDBBQAAAAIAJE1Q13rES5COwQAAC4KAAAuAAAAQ29kZS9leHBlcmltZW50cy9jaGVja19rMl9jb3ZlcmFnZV9pZGVudGl0',
    'eS5weZVW227jNhB911cM9ESmshIbSNG66wILd5+6aArvbvtgGAIj0REbSRRIKtam2H/vDClpbccJUAMGb3PmeoZUHMd/dLU0',
    'KhcV5KXMHy3stQFXSmiNfpIFuIOetVo1DnJcG/EgQRWyccp9BdEU0LWtNHCvu6ZI4ziOor3RNWTZvnOdkVkGqm61cSjbaCec',
    '0o2NomEvt0/j9B+rmwBthSsrdT/i/sRlFEWbu7vPsPIrhspVhap5aqTV1ZNkPG2FQafsdrGL7r6QpAdcQ7zWhYxpgrJd5ayf',
    '71Uvi+xxkY0xxYRK68dCGTaoWn02nUxA9sq6TD/6JY/WqPomvY02fpwvbqNP2V/vP3758Ak32E16czNPgIZFGG79EPbmtzy6',
    '27xff/yAsj/DFWz8/xrmP2KEhdyDE+ZBuqzS1rJ+CftKC5dAN82wLk4MKw6zX8NsGQH+jMSEN1CLnvk1/dBoMi1YD7OgAX6A',
    'DYerK1jgDutebl8EzS6DZqcgPoQie5G7rBU+ocoswXVtJbdDIH7YvR1Qj1nrF5gqizyQhVfD/UmdQEnp7ufocr/gmEJMN+vJ',
    's37ul17uGYXq0VG/I6yVRMZ7y545gkt4h1TByVzO5ovzPLINgks8DfJDmMAWQ+1KrwFHPAE2R+FnnDyjfeZLyzllw+iDRUe2',
    'u4hay4JqYORMCNTHQk7YIb/XwcRgh/ZD3JQBkl2TWwmsyTw/joscsj6kEIq+76xDxFkxEBpgsilCb18S8YbPszaonE1QDu+O',
    'kof9I1GNNEabIWjaHu4UZAsG7923aGC0sZzYNpWcXJh2qYzrkzLSLzfYJap5wMORvc+hDFSUa8z+JBraivxhE2eTSUFyxP4J',
    'EgIh/iAPatWwk9ZEN7sBxpPTrl0cHXEfeEdBDy6cGxgylQq8RZuCUYYHy7OXFRmUntSE3DtRdVoP4t6ofDL97zSjX2zjJZbj',
    'dK8U1T57EC0elWdH2ogcb198FzJPgAxbVhsUDFfbmXTgy3TPZv65yOhv6MlAWJC4DJN9Lq19FYRJumj0BEaWs4MqXIk4dgbk',
    'vtfSWySO5WdaMLXZ+NSF5KKCl/n+jvo2XoAH5Upg9A5demtSfPZinmosCosPcQKNPFSqkasY57LJdYGkXMWd289+ijkWGui5',
    'C01yMMpJugNQR/qbyt3ffoORBN6qSlZFI2ppVxW+Woyqv73ZDYwJ2NQPpRQFwl4eEMTjeGS7uhbmKxoLhIlzYSWRpULHvUgI',
    'PRZVlYWPB2Qr0pIK5B/P6O08oo7tpdOdbxs8pcY5NhQKOlT3e10zIxp8wpewnUpBHevVv0aF12x48Ojb/wXvBj/xS4fcie+a',
    'ij6SQOSlkvQxdfSx9AsI+H21IE8VGoRKH8YjUBaIHWmcRN+iV3mUDQVK6dsJCeXrlznZO0Y7adHVrWWDUIKuUpZXC36BZFFr',
    'sJPfhvHoP1BLAwQUAAAACABRVUNdmQvpcWgDAACkBwAAKAAAAENvZGUvZXhwZXJpbWVudHMvY2hlY2tfdGhtMTBfZml4ZWRf',
    'azIucHmVVW1v2zYQ/s5fceCXUbGsSk4zdEFdYNgL9mVIkaafikKgTcpiLYkKSbWZi/73Hl9kO0uKYQIsH++eOx6fuxMppb9L',
    'J02vBmWd2oJQfDfoIO6MEhYabeDur7+XVfmThU5veQd8ELDr9AbFjZ4GYQtKKSGN0T3UdTO5yci6BtWP2jhED9pxp/RgCUm6',
    'T1YPET9y13ZqM4Pf4pIQcntzcwfrsGIYUXUYLyuMtLr7LFlWjNzIwdkPq4/k5r1HBocXQH/TQlIvIHbqnA1yox6kqPer2p+y',
    '5w/UOxX9XijDUqT1nZlkDvIBWaj1PiwzcouRy6JaXZE/6nd3v/rlL3ABt+H3AqqfMVUhG3TbSmvZ4RqaTnOXg0hSBss3Ubom',
    'gE/rU0XPl7AAETRGIlsDYFaM3cISEQvgG4yVZXBxAasc2Cpt6U2t/0cLsArBBxQOGI6FjDJ0WULMFBPbtnK7t/4EBM9d245v',
    '97gK2TCqhoZmxBdXHUANYPiwk6wqqyxmekAoQ9MSrsosnXdVlsEWvITsHH/O0z8ieEfIeYSXKYJ/5oRm8pA1nz/zuMtwxMhD',
    'IkRk2dH1/Dwos+M6j1FPSG6txLaK4DewrOSyWh2tiaLFGipCYkfXj2lT/dTXyfIsgzve9xyVl2d9cbmKxLrn6HEIRkvqoHNG',
    '2x+x6btG+cqHDsGmOjnOZLdoCAQftarBrV7P7eZHNrI4616d4gcq9ODUMMkflcel8oTznhfiOYZSTZ6Y/md5HlckVmkaR2nq',
    'LbfS1+jDx8D0vWeOlUVZVjkObHpf+XcQcYITnd7p3pN/HEfkdAE2DpsfJT9taeS8PuYadkXwk1E8DdzZcSI68HyBOy7gdKyz',
    '9AuO8iDYV2pr/Rm1hl7DfQ40YiLvqArLozogvyjh2tkUki6u/FbZt4wQO2GFzD+Y7dewJQ0f7dp/zROV6BmFPALmUkVgKAZC',
    'TkMVUakaj+M8KtG/wp1XPsZ72hDR44yV+VRxley8m7eoR8+xQFT4YEcz3i8SNfRP3MHJdG8FPLhWYm/I+4l3yinsmc3kQGhA',
    'FxgNkhkQqfmn3q+0kX1Bc/KNMH+34P3h2r4q6+Mtkggu/CVGs+KLwU1rJx8c85pCTP1oWQLluLvAC2a9yvB2GbZaqGG3ppNr',
    'lq/w6zEaNfyHW0a+A1BLAwQUAAAACADJNkNdB7qh9P8GAACoDQAAKQAAAE1hdGgvcHJvb2ZzL3Byb29mX2syX2NvdmVyYWdl',
    'X2lkZW50aXR5Lm1kfVfdbtRGFL7fpxiJi3qz62x2A6hAXYm2IBC0oCZtLyKIZ+1xPGB7lpnx/oRwW9HrSr2vRF8gUqVc5wHy',
    'EDxJvzMz3l0I6gUoa8+cn+985zvHN9jTBz/GexM2Ys9/fvYcf91lYskzy+xCxTMlG8vetDzX3MqMZWouND8RjDc540yraWss',
    'a2czodlUtU3e6+3sHFhuW3OX7P364AdWKM1sKdjhI/i5w6zmjam4FfmW3YLXslrt7uywQ5ysuc1K2ZwwU/OqihcytyWrZSNr',
    'vmSVWnTOmDRMGMunlTQl7Bkx47AnqhWTjfM33tvt9W7cYA9cRuvoZS4aK+2q13sol0w1gi2UrnKWXl2kQyaKQmYSJ2DFCj3n',
    'FUsfHF9dJEdXF7EeXl0M9AscIwiAEVM6FxreVWtnrWUOMsPS5fE4qePy4x8flseTpB6UKQXlDOHyQiKlFG//2UudpfSsjq8u',
    'zgZ0Qae7hEPT66VX/+J89BbGhjDzrs8SoLN8yyIdlwN/pX95PmTRRJeD8vK8fzSOI3qMpyN9ef6CvYOx6MkkHvd7PQK3kBol',
    'Q161x8O44iB/I7YyF03uEnGBANhMebjp7BrGShnDuPUPNX5QzSzXJ8KyqbALIRr3zqHk0DG77HGBfAEazyoybFnJTQcaDhnU',
    'Zo1SVsnZzBktRU3VUOEVy3jTKCpQpgU3m5hAXp6VjPiFLNT0lcisnAuWC3/QsAWAZbWak9lCq3rt1KoF10gScTSCaxDri3h8',
    'rxoj3rR4CJpRdlM66fgTm5nIJK5s9U5RKfCfiEoUrFa+ppOXOxFqlNxBiUbj28Qma7lsQKPpiqUHyPHlTvKW+Da66Rg3utkV',
    'ctKnJnuulSqoYX7T0gKx04Sq7rlEXVALlFieolO7igDmtE1gaw536fzj+/dHYDOo7GqSLsHFFm8o/wwx57IBiAtpqB+Xsm5r',
    'VMlIh6YrPHJKo2Xc9qPloI0nMDzR/fQek97YB2/MH2vjZR8UjXF8gJN9+HzmuVGJAnUqXL/ggl0zNLSTa3vkQdSDKU856rxy',
    'E5fKslY7JqbzJNYwcyJdfVPXJqfohY1DLU/KzuMkeAz8vuYS4f+vr89dxcHVd1vkD8TfTs0x46uQTEfhTVo+Grxfk9Z1odTX',
    'moywnR/vJKfReFCOgD6EpZTEfwl70+tR4FnGW5Q0PTs9I6WJS4R70E4N5LC1UjWUD+6uPbkY07W8ROP41CsL1fCQvw7diY5D',
    'RMYB61q+1EI4lUFza0UmvQhBj+9bouvesEzAaoQ8VdBCfxSNRWq7aQu6Sy42krE1bHbZwzBbtCAlE0Nm8FoZ4R52PevFDlgZ',
    'qyX1IFDAFNm48WoLOf4WAQ2ADJ48BbypTfyP+8Zp9Qeqd1pEZT8JeODZPoyEaVAraJJqIBl25RhWpAFOdwkNVkTkwdKfwfkg',
    'uqVHk76Fl4eotZzKim6Ha9Y5GH0NB6hGhrqVAxsmxCNBDz7V3pAo2FnBmIXYkGMUzYaiMXhmn7mO9/GWbUJKUaMDtTbi5qO3',
    '3IQiAm+orAjjoOKyhnB5paNpT8VYscKlg4rNuNTQzU7Lc8Er4wdgJ73r4fDxr7/dvH4cBm/MocnoCOu2hI2qfrJxPG7WuwV/',
    'DYj2vjEEkUMtzZLxaBLqk4sCEgtQymOTRHpgaEgeTUBu/EkjHcJrkrdZjPfDbID/3xEWIIfxQyyMKXQXYeHTRCC1tLTOuBmA',
    'tr2+P3iipmfQ84zazsBXQFLWM2pWBAXqQvaZn/gIpJv1JCzHxgU7BO3wJ/5dnr9Ltn4Ee/sYDV2Hf6ED1krjtw8fTpDATzm0',
    'Vp7hWpUceNJJBFEbblPXrtbTAOqEkTkL49apJae+ak24TXZhwFAh0Yi4d+ImnU0OyBRtAVlbt5jcZEAsM+GXi1o5Hh8ebaUb',
    'B6a+AETRGCwe3+6bw8GzCO8O+6kbdebj73/uUfkeYTtj6S/Hh0l0C3Q3h+697EiDzlJhEvGcAPDcor1UOYGl/kv2R7dvOp2Q',
    'bvBBjtAAEgzIt7k4ZLSU8M0C2y2tWYmRnLmJTHYbjFxrOlr7EtEq1Jqup1QNMdnsn6KZQ7JmtKRAkTNOEz+jWWuB+ptWAnq4',
    'pgFtY9ROGipTIUQ+5dlr7NlwjCnilvMMZrpFu9uDAYhpaxoZynfl1iJLZCe+SD9IMP20W4iukfzeZl17ha8C7EF+gpIi8M8X',
    'JC/mblxVnqvhW2JNDTfCglPnwhC3KC2yV1XeIjTjJ0QOXQdKamZpcPOOpCQ8YWnB0gafrJQ5Ug5XuyUyExrRruh7BmRXNPrz',
    'mFcnCptVWW9/cNwLHxXdSCNl5A2vVpb8Y7BQbKbtoMED9wXD3BcMQr3PTkSDPbWK0ycpxdP9hIaC7R1bZkiHZlqNjRBOZqLZ',
    '7f0HUEsDBBQAAAAIAMgzQ11f02OWdAcAAEgPAAAaAAAATWF0aC9wcm9vZnMvcHJvb2ZfdGhtMDkubWR9V92O2zYWvvdTEOhF',
    '7LHlyXg7i25aXaRtigbNYoPtdFtgkEg0RY85lUSVpMZ247lvX2Dvi7YvEGCBXOcB5iHyJPsdkvqZtOiVZYo6PD/f953DD9jF',
    'l/9MHv7jEZN7LhzbqL0sEiOv2pIb5Q5M1U6aG14mqt5oU3GndM0ao4Rk+M90LZluXdM6VkihLN5OJicnXzvuWvuIPf/3v/7z',
    '5HO/020ls1iWBdvwSpUHxutiMK9rrGykLNZcfL88OWHPpGP5d+nlw8XZi3zBcpGena7oweDhIzzQ5/nDT+y7n3+jhSV7zLaq',
    'KCTc44ZXEoZZfvfm3U8/XYrELsTcvsiZsiFE7xIvS5Zf5Mzoti7skl3ARbfTrISXtWN6fS2FUzfSMm7kZJJvs7OXd2+m+1k6',
    '3Sd3b+Zm9tJ7tM1W4/XEry8nk8eOyRtpDuEAn4FSclPDMSth1acEnjKhq6aEw3062KatBaXaekeDFQSTP4Ibz7Jrf9Zxn4js',
    '+pjYWTZ/+5oc+fL+m7md0fpksttKI5HC7CwViclD6kS2SsXc5D5uKylG71Gj4cVOYaVStaraym+v+N4/6w3Fex3izZmGZ+Er',
    'aSrlqLpD9rto/AkHf4DQ9Y3cL5g2BVwqQhWvkGOGfaqgTFQA2JCHiIgFlkTZFqq+8mWTdeH9ZFeGFwrVsqe2Xfd/Qi07n4XG',
    'capGVdlOFW5LKMi/Tafnp6uZzT+G/y0Wvsku4tIFcvKpxr4/BQJbJdYZXV8BsCEcHwR3rNLW4e0z1VixVe5HsAMQzsn3QjZw',
    'GcbwEaXQEj48xO/+lyGXr/a3yOY6VMBT8pzttLEucdxcYd9znOx0YvEoKOn8CmDS1pI1zizyUvZM9JmJBIH5s5cn07s3sxQM',
    'zl7twYfvbkeH+lTV/twgAb7qfI9/DaJGQU9O5F5Ia0FKZYE/svOY2bbJXt2n1y17ghd3v2aOXXYnZO52lgxOvGApu5iujJ3b',
    't69n+DM9Oz17OF/Z0/MZQwUoLU89g50SpEFInrPMkO4wp5loK6xSLQaMhJqWCuVZS7eTiCYnqxHnfztd4XG3VaWMIkTFS0Lp',
    'SOJo11CztRcDZmTFVR3EIqIp5oWXVxrSuK1iugEdRJmKfETVQHgyXAPAltW6RzL7ocWOYLLUO9BkHTaX+EcAWytnONnA57pS',
    'P8r3VbJ3wPoTcGI9KEgDDCadFPc0STo96d0g1EsFDYwyE+MdIZWQRfmKokrqsyRtf2603pBCd3LZ0yOkCB6bQJSctDBWgeQS',
    'TLA6IG2zUYKYygjQoKOHHmLLL+mbBe1+AWh+QSod0e2ITYAsrPgytg18JkUiCRoMBr7Y+6JFWwYVsF5mWieX7AkX21EA3Dlk',
    'wfbKh1A0MZswqHf1ECASBS/wZji4K9KCgU+C+sXAshSevOqaxtvXi65PvH19m07N/Eh/jyTWBH+c1dYKMBkfRyhLUYIFqSWp',
    '4JjZhr709fiTBsLXCB/YJHyx9aEz6pW0y1H469P6Q4vSFaxQaNY1dXmjK5ydE/vc+IT8Ej2GmJ9dU2sdI1qCWqVuJLn9B4Ow',
    'wzvh7I3x2B/bpnnvezcSckIkANeLP0wXJA3CemStSbOHLrNk34ImPleHNLbEAB7gPoiGhIUSqf3QHgJ988O7n3+3EbPTQ+ig',
    '/ZvfINvsa+WjGLkFzlm1hrzksEIoDUOLLxqGlrlJz4eJpZtXFvdMDH3pQ4sm9NEsPbcYdzDUlAgsP05Dz509QNc9wsJqfFC0',
    '3O95kPrx44s4dY1VxrcHo/Cm4ja45y1Jz4MN8TRLMCTYnrbZHFMCxf2UREEJFOsP3RnGvger4pgCNczfFxLILw7oC4fWVhZs',
    'x8HErbayXnQqVheKUAvbOvSkXuweBAY2HD22k8YwjZKjem3h0kgqbRCP4FIY+XLf+2LinVHc90xVexBQI6A5hBrc1PMxS46B',
    'mNn8ODtdMQCDWXwdOkk+nYZNAMg8PM7xiI00BApaRYcjMAVWX9AIBhzFGYtKkFSS1zRBJ6Fd9H0dMICyYFREqDm4PY/9ksDX',
    'rtEdhOvoGucljTU4tSk1IdX4adC2VUXbvPhhzA0o4f6Mfn1oZiGnNfo8s1vqRF7eej9o0A9pfK+IW26HUWHUnsPMMApkaPuI',
    '4zNiqLGyPIRyBNNxfun6KZnOj/hD6ujZN2qxw8AZ/IgKEpM8nO6HsrzLYAC6zzcqqToJ8GOAC63pXnh+kkQ9GqNv4sR+f0pS',
    'nhXu4Etj8dQGxQm3lEAimizP7QVRzUMsRkwo94PNkr377y/+2iQgeqG1KmrJDVdopX4c/Dvwz9HjcUAhEwsa0jEC5YTfXp++',
    'Ss/yR1FIMLZi6oDS+gY2nn38BWx0vwPlSJfRc/Bz6Oadp7jOaUlzCzocXqxLZbekFl/lrDubx9vi2Br8IPBE7ITbThjZO5ip',
    'KqYy8rsfbfiOlDtYFJJWP/7LexH6icAAHOIKH+BOk3vfwzBjQ5/BvNd6cBLTClXJ2gaFCddQ3zlquFTGUXBtNC8SUZI+8gYZ',
    'F3ytPEIMsKLgpJV0x4Fb8obKL+Ry8n9QSwMEFAAAAAgAGzdDXa+JLH6uCQAAgRQAABoAAABNYXRoL3Byb29mcy9wcm9vZl90',
    'aG0xMC5tZIVY224cxxF951c0kIfsbbgXiowkawwosAwZVhLBYmIDBL3TO9O729HM9LK7hxeJerd/IMhjglg/YCGAnvUB/Ah9',
    'SU5V9wyXIq287M7OTndX1Tl1qmp+Jw6f/imZTh6KpT5XRWLVqiml1f5C+DOTbIyuvah0rSt5LnCt7KksE10vja2k16YWG6tz',
    'tbMzGLzw0jfuoXj+3V/+9uQrgSeEXyvef/JAeCtrV0qvCnHSyMJicS6WstLlhZB1IbLJI/fx51/s+H62OxiIZ8qLzKZT/ByJ',
    'LMfFjC5+SI8mo+kxLnnN1fuPP/10lCdulA/dcSYWKrghFo0Xa10UqmY7ZFmK7DAT1jR14XbFIewiW+CbWfxd5V6fKsdb5qaq',
    '4BS+NqXyqnNZqPpUlWZDj1kl1LnMPSz3a+OUMMvo5RfksbbCeWvqVZIbLDqnWOLKeYnjZnzKM71x+Vr7V6LZbJQVC7IL/xVG',
    '1MaLfC3rlRJn2q9F5rJor5K2xrOm8ZvGwwzgYpwnmATD5ARtxS7yIWbhpa6dMDUbqlrfOpeWTZ0ThE70JK+5EIWyCEUBb4sA',
    '/crKQiNOrj9i02LUsLxRCCTDdPXf+ezHQe/qfT99YD/8Op4eAJ+NNbSRrsWzJwjNLCBW0vN/nR+mvf3xrO8Os92dna8BkDpV',
    '9mKbA6OdHXpufDARuINtlvPHwjWb+eubmL8RT/DH1X/mXhzBjqv3vRdz30+uTTrm9bTVbIJI9qaT/s4Ox9OcdZEHP8yZE6Bo',
    'YSr9StGdlUEWrCvHmLZkAuMyoR0QAu41QpsrkA4UU/yUUyeNwr2A2Da2WCOBuKKYLAIcBfhlKbUcUmEklkoVC5m/TJZWKbGR',
    'oFH2Yu7S13mynpOz+HxDOYDvtGeHro9QH816M7o8ZpI0jje+lcnZt+ks67JYnefKObGWoIYF4HDqnz0EqJ8JQEZJC2PV+abU',
    'ufYgYQEWWiLkrvizESsFEkICsm/xuO1+5qV0FECkzNJwNHdJE55bY5acz0wCsaJEG4WcrMPxCEh0Nn1dJetRNVy/QYxrpwsl',
    'sieAFE6HVHiVVgn9BDci1v0UHr3u2WQ9vHx1iYiMBAKyHq4//NrvTZNXCBEY2X/DyE+B/Dce0Doj3JoA92tkEYXszNiySNxG',
    '5XoJXeJQNRWhlqn5YIvW0lNSBRABz9X7HwcpKJnY8b3R1fshvt4wFFhpNl4TmSxtMxg0tQY5EAqJJFyJpt723lGiZD9kD4Ve',
    'Rgwt5TYIQupCttMuC0jQGZsEU/AY1qy/xJnkfNTEEOjfAwvKKaeSwL9cljnowIpdyZfKRb5CmIpwDDRLs6atrAKMxANAxGeJ',
    'MxxF2wGCjz//a5J9wXcIkAmOBellSTyzuNIWe2frFFYhEN8shQJ0JFtWr3QN1Qn6Rd7g6gbIeak3G43YgHbexNviTMmXsEpX',
    'rChIPSKpXKkRizzrGq1raUS0ht2dgGUEDlM1I3yyoPKyJjFjw1sY4dSacpeJGS3jHWEpPUy3W0TB7e9bjILMCglENVVFCEtO',
    '9cJaE5N/V3yPNISfPiWcQukq0nUSY1SzF7XSq/XC2LUx+NuzDt6jfL8sLltR5Aej2JdaMWk4SkFwOqdv8AZkblwEUjvkTm8P',
    'OyfF0CNfEv6BC8jkW7qe9Xs+KfqtMHN8rXGOcKFtR6JBxctiksGwPeRG8AmawXUklKDIAUIyhIhO3spNH3MzgWF8+D4dXiR7',
    '+INOB3WyAvtPotudc9eC6qkoAoKsNXyIWPUD7bLiy0kb6keeu4f/u0vcpLhjCxjYbYKgk5GwLQTPxytYjwcgLjeCdsdBlrbD',
    'aXvXp9Xco+TSQR635K0LDj98SfJ3Obxk5lz2uz/Cfda4WX8LN25iKu5ylkHWt0kWAxKqwbZ9dHswaCsAFCu7epcSzOO9GQ75',
    'mmmPWryh3mdhoMxtT/IJJymGCM7bQGSYs4B03oSBs7EN1VZJdkHyz8hebJztg/4Bij1OBGRwrsqSVmUFZJMqSkyaWXZnBKfg',
    '1/Sg75Peg/H9vr9mPH7QZedgCGA0nCBfIip6oVnisvWQT8m2VPRm2m3xaUYZRtuA0R0x7gh49odYX2LIHnHitMT7JbnO/c/R',
    'd+9zm7y93uNzzNwPeyRxr5S+7re9RWjSgiZul2cW4lYyPbdWpIA3BI1kOXp+fdrVu2BsUFyqCiDQ3XUh1oQglOjAsVwtUaqp',
    'N+2iEgtvrBjMJpyKvutWs89xoboUere2pjCNgpvXu3tpV8oHhb2r1ETT5e1aM6IyiWrquLeOgftchG4kU1eQKRMoq46wPUh8',
    'nxoN+j7O7jCKmlO09NyI0uIa7T8fRh7/Fgqhx+1KNG9UGKK3MciH53APYx78qqjLy2gYQ+8bQttRknso8nyepGjPqXThepii',
    'TSfV4JbvIs5nsePDM+h3dMT9t2MD+eLQ8BHoH0nnArWDJs4TqCJfDC/74xno7jLuNYPZSaWoJfjUcevGe4GA+kY/cqsac7va',
    'LjN1bBdhO81ctw7QN4M7ngXtPLhHWUhHoo3LJVfRbtoBUXS5rQ2ArVYryVyNcTHcR/HRMBrTCMYGBOr2qNpOElGiAp+7gQbt',
    'bWQsGh9PnXahNgCxKxOqnW6il1gfpujt8aiQm5BH3Zy0Kx7HJOJKE8SFBwqmSZslYdrqJvHH22EdDJgdqDifkCrwFt05vT7I',
    'm4o72VN1F6SH470UbcZ4ut+nOTPMkNvdzNZUNqJ3Fc+7uYQOcz4JR+KhJDQy8RA0UC4JnTdnNd9I0dDMHY1m1LQISpTsJHVj',
    'C2CnXcl6muKBsU170+FJO7XhivL3K/TLOc0kK7WwMtiB6nWSHM2eDp+iP3sAGT4WqTjp3Uum+yfJ9OAEd++dfHjXHx9ND3on',
    'wxm2pCn37STb2UFWZhMcftIa0GM2NxWg8YjAxpQXNUAETQqVkzaFImaDJFpV8TsDlHYd8GWHaKd+rAJtHIi6lNjfgT1Ag9pD',
    'mihkhQ/Pb3c4ybkwd69eYqAJJ0bpGiSezj/+4980NT5uCu1pavwjJb3vhvXuLQqzqKg06tiiVNszanw7E1oX0Cu4zokSpuDY',
    'w4dJKLb2yMWCqouul8punRe7dy4quItKUYmXSm1ouOqG69YoF6QN/XFd6pfxvdcBBjGJ/NS+gbC4MywOc0sbaNA+KTR6NAd7',
    'ZDkKscOlvQjznNfBNnrNc+c7tOsXWNtvHVA88Nzqgnxz4mVtzkpVrFSX4Z+8CmrfbvGRNb1QQiKAIJrM6gQFKt6UZeLVeawH',
    'ibSeBQihcHiQVi+skRhs4xsBuSEoZOyegteIEAq4KU9VsbvzP1BLAwQUAAAACABKVENdzEXH0NEGAAAvDwAAFQAAAE1hdGgv',
    'dGhlb3JlbV9pbmRleC5tZJ1XS3LkRBDd+xQVwcaeafnHMAzRsGjshukY//BnmAiCsEpStrqgpBJVpW6b8AKWrNlwAE4wR5gD',
    'wB04CS9Laqk9bgPBwmGppcrPy5cvUx+IyxkZS4VQZUY3Gxt3YnIo7sQ5uVp7XFx46WuHi5FzdVF5ZUq+O6SKcKBMFfHtiZmT',
    '9rdCFZWxXpYpibuNuyiK1v7ByeXL42h3DyeProRyArYqo0ovKmuyOvXC2IwsHp+dn74ec0CvpVYZgvRk51I3IUxVqZYBHZnF',
    'UExNXWaSf5JadG72+eXbUhYqFYVqHZ1+eSgSfl0slJ8JvzCRNmk4i18yP3Or3kd7f/306+gj/sma7yjl1wYiNeWcboSrk9zK',
    'TFHpw29eqrLADd4+NkhDehLSCSkS6UirkoZCq+9J34qKrK9t0niVNq+bY8vAP4SFSZtxZEocmBJliUy/R6Y2BfLx1fXlzn4s',
    '6KZCUJQJS7klD5zgjnNaGKsZNtcWZSWlZ4MOzvvGBwJvs7GVY5MyU86rMq+Vm8lEaYVqw5rxiFhJjeMprqzUAyTnOenakpiT',
    'VVPVwlrCPqx26T2DgdMS2AQ44S2tLVIKCdWlQopFVFmakiUOok0tlItucIixnmqVcljCJKEsc7pXt4sHiYx0TomVTPFa42VT',
    'eyFLAf5a562BpVRLVfRRctG/YKJRlNRZjgjOJOIwkSOOAOdkTi2VptYUopJAQZPuuaZJ2hL27zNqIEYfbvbEl9ZvDUJVRh8z',
    '2fZ2RcdfCdvL6ALn/gH9hBPqejWakZzfhmyOxm3LjQN4e4fih1pmKBTaokmp40NwuCCVz3xUyEocqcqlM+V/ZPcMpV8Dcm9t',
    'ilbTHOCBKSpNoT4Ojy0NRGFK402p0iaF0NY47oS7Lf2M+HgLe180BD86O2uCv1AFhEmWZKBKiCZTbbtXpqp1Q7QE4TtuETF+',
    'cxYBSG1yZm+KC+dCcqhZ5FIIn8iR30oyX4NeqozQk+lMTKCFqz5gu5LOD5tyI2dAXjomKq6nFDjvhgjFKaaiyK2pK9a0pGmZ',
    'RixfGppOM4AyEEHB+D2VDZjzHHxgEjKuKr3snATPc8iGXNFgwNi2F3zjnauDiUhn4IhBtrc9f5/DVpCJzRe7r7bivudVyYE3',
    'DrRZQG5bEgO3J0/oBvrinjxZy/YgL6vdl6iu/9bxoEF3gBYGi1UQKnSZA1SlEQVDzUZyQpMA5SCYD5Sv1rofE71Uxa9iwbGh',
    'j6vaO4b0BuZnKsN0Eo38dZ3cJVAQx7G+hVA/q4yN0JD39YsnHsfZQfsxK/wy+viP3zYZZsa4QFkLeSMaEHu3zVRjgEH1ZXn+',
    'FSVmvSzCDGmODFFvijKFaeEaYmamwNAZinhfxgJdwGOkb9olpVqHgf9Lhe1VikkYuhlymyM+cKxFZCWiqI02iOSwK1lXw1A7',
    'A6R6lF6wgLb1iqaWluhHJcqG0kXviWh89OfP168232xdxiugLD31WeVk2NBSZ1jT4jcxT2DArEoEslCOlmjER3HUn+2nxQMG',
    '/RNbPpcO/1vHuOoHeooZgiIEtwJzzLLP1k4O5YNkEvWYfNLJcPzqs724IW2ECQcJs6yM7zEovtzct+6pe/d267PNvZ293af7',
    'buejLfAtXk+cIEz6cV0Oi0sn94RbjaqtbZ8Ha8IjnQhOthiK5tFjQA7vbSZr94pzKgyP8iVjMUaUrzPWbAwMHoNyDrkNrRQQ',
    '7KnI5WSGl+1K2pExzL/9DniOoYmzl7WMtQBn+LyxMtUkptqwj0/su7c7e88fQbutaYfv5wY7SjDuWCsh95h3zIglkiCMVlXF',
    'mczgLCwijb878VVXry4w2S0tl8bokPUDxgCGfc5c6luHjZozRqRnTco8omGwK6RcQNXbIy3F6grbKIj2DZg2u3ZP8Qe2RW3i',
    '3/7HzGP31y+/250X6EMA3KsydB5T0UcZpYo1C6yEtIZx2NVlhN6kOYw2odQOl8ltcMHLUNs7YS/6Yn3273VNQn5BqH4Yf893',
    'G140K/Pu/+ybsM/Qo4vzuu65hwMWdc27j4h3P11C1WGw0xYMKzQ+RdBk1qLSHHSusULodlc5eFSV6cY3I+F+O3QtMDl53WxR',
    'k9D/TBaT19Ts1FCwZhOXCfNeYR+OmoHVUfNkPD68EKPDw8nl5PRkdCRGFxdXx2d80xnlfTFlPfaK7QTTdcU73qpit4stViMz',
    'xWpLnNcF8a4l+fnGxokJXyFY8mbY//kbUXYtHYbPthitfmnIGlsavwaoeW/kST1scSzxXQWgcDVv1s3wueEAYMoJsgqJeHt7',
    '5wBfazuXL8en5+Pj65Or4/H55GB0dD26Qr7bRRZvb/wNUEsDBBQAAAAIAEpUQ10pXLsEOAkAAOUSAAATAAAATWF0aC9wcm9v',
    'Zl9hdWRpdC5tZI1Yy27cyBXd6ysKyGIsudlqybGtiaIAGkvQCJYtZdROAgSBWU1WNysiqzhVRXW3oEVmN7POLtvkC4wsssjK',
    'H+D5B39Jzr1FsilNJsjCRouPuq9zzz2XvxBXztq5kE2uw9bWvXhVSl2Je3FMF4RTvimDuN+6T5KE/+GR6ddvkskentnZufrm',
    '8nenJzs7Y/HKVrU1yoSl9kpYlyt3KIwVhc5zZYQ0slwHnQnpfVPVQVszFtfB6SzEh4X2IldzbVQuZmshgyiV9LhplPDxuczi',
    'SW1kUGPR+7H/wA/RGDrreO/zX/56/HwspoUStbN/VhlZFLUN8FDLkqzNrROVzmurTRA+k6V0+k7ScyM4DvuSHhYLJ3ONH34s',
    '0uX7kNKrN8YujZgpHIHzS7mO8UsnAx0q17iHyE3u9aIIiVclHFB59Gdm4aQwSuVeLHUeCgpWRgcl7mT9SYfIP07PtFlQTtJ3',
    '76epaLzyQgrf1E5VTSXsLQJO/5DCvbP0p3E1sz4ArkcbVWbNrVrpsKY3fNUsFiUSr1GUtEqFrGslnRehcLZZFGKpKA7P7rHL',
    'MUOSrgi1qkudASxzBGEdTsjZD12117UR6cnojDy8oghk6a3IZB0awAvlUXPllMmUyAppFrik573FW+nWI4Hn6RzPVkMBPNV4',
    'Vyhzq501FQJCSXOn5wE2pkd7qVjoWxxE9o4m6SG9I0IDbCU+qBrRO1ui3uu2DOTebyZw8PdcjzvlLFKfUWqRnYCwpctFLXHv',
    '8uxkgL5nD9FHqUfWlbuVZWJNuRZznD+T2c1YfGXx9tK6EvbWWpXIdk5gBPDEDtW8VEHt9K8Dbt/Arq2M8p5Cz2yDR/IHSNOE',
    'hLleoXTKtBWnGhWcTs5zvjayQt9Z9BzgQiHCj4iCBHYa1fvIWWo99Ipt4JA5/OEutDNPnhFOY0KRGIBalHaJJyKquxI5aTyq',
    'KmflsFd/+Ygz3shVhK8nrKgVgs/KJkfhcBGZUUjCeQAtVNaAA2CN0mDmZWQDsMOSgqG+OIqVHhh7/t+IAadYQAwM8ZIYYm8y',
    'FqcyK9DkOm8xJxYKuNP4Rz0+6MuujwY1vebjBs3EvxzezZqqQZKBwpZaxELWaCgr/LeNdEhbe3KuCV8I9VAcH4gWcgr/r4Wa',
    'z9E/ZDFIB68ifaTofCRqnm5406kgiTjH4hi/b0HAuXglm6xYJ8l1ViyluxOMe6TZWWoMKp8FBNNPf39/8/n7v331/iYVKHRF',
    'Hqa4QNcFXR0JmGsvvk7FkpqPaw3ooQVbsuI6p7hPiUo//TMlVrAYBGQIJIZGpyaliHd7uM2anKN6i6SA4kqVtDnhHqgpqvgn',
    '0UCp81jci9OfTB/uOw7JqCTX4AOP99BXfm1wmcYOkp4DQPg1RzuURNilrmtiVnJZUuUMk/QGjJVcgGNQnlmpfYGI4J6Yo+Lh',
    'UUgP2iG9GH09+vTvlNPTcyMZ8U1fUHI4ojQWFdaDNo1tfDTwBfqoqZRjdmBQkC9E1UGXiEtWkQsoIWfK4HYprmAw2MTXKtOw',
    'gxlgEMJKZO1E39m5vDp9i2zN1NrCHfKZfXixO3nZpyXSGuy3sM0biiAWIrcq9rf2lhgETZUB7cAeoA+/q+h80jusnKOB0Pfk',
    'i0HZRiQueLAjafDR+zF8O+4EA1NIGe2SPukbn4ehtjOe6WivTWVjAnoG5RlkqL/wSoMKEkpbHuj4lk6l/gubtzbdffkITbCE',
    'KkWJECkcnbrKiJ+9CpsyER/tPjmYvN4GCGa4yqmeKeIrYtdNidAxSTunqRHO+xknISJCVmgmlwr0l3CGBHHgYZyTdC/Kh25K',
    'Eyh0AH+2E7adpk4tGtI2kZ44Wj+oycv/oyZxxPSigtkSOUj/OBnt/akbto3RgHXVpQSipiKK5myAGboeBXZVi7bRJsoID8Cu',
    'I5EYGlNcbJGoLpKDyaZmoNTIzLEDBjlCHBVgiamnzCDWg0GsHJlZD2pQKQodkiAQ4pjZ0PCQVnHMEP2jP9Mfv+siHpB8X/w+',
    '7Isfv5vGqDe6lbVxepEmF7r2CDzcdXqsx7MfsTXbDOfMtAAuICJZ6WAyzCQkJRgfKopkqKjA9VBxsiwVJRkXI7l8QT46RwYa',
    'SLp+Shse6D22Bxn68ufQkL4mWdVj4hWPvE3XQIqpEtnmAZ0pXJF9CxOWBs37sw0b6xxpNg6EAJAN2S994p7er5JP/7rf/vgB',
    'sq4TPQn3lZD8ZMfpBLYI3EyRhQg2KlA6fbLv/FP/8cM2FMN5F0OEHJl57nf3U0EJc5ywZaFBHW2pNrVjUmcJkDyQALHHOgyy',
    'DzG2iP2hT6zgNsBjqRuo2pq1NeY8WooZJ6nxjGo3slZ/Q5jk0NGATYIzCosU5LewjXQNhuW+2BWo6NXDZelxdREwY4Sqe8o1',
    'GGaekolhonnjmD9Um5mznuqVRJnCOiKuIXCQO1iReNaOkY2M0GEbcfOw/I8Ycl7aqHPTL93HD7t7L1B1CfyXpJlynrl4ig+H',
    'jtOK2rIloPTJ3nOHN7b99OnlExR7uj3Qh3uT/52LbnpGkUXBpJNf+88//MPtHqSUI3IWlPdtQ9uuzIAP9iIyg4QqptFNTYoh',
    '2nrE4zD24Aa7u136wNoLNXMy5rRTJ3QejluUFkEz46XPKBPP4CNIwmOJoI4PQA/U68y6wloWgmFpk75MwA4st+KPGfnFBDy7',
    'AD7BVY4XDX3HG+Bwe4FHFoOjqKBOOwS1mxWdsU+Cm/eaDei7zovN5GgEPVBCscrtd4VeUKgVWisnnSndTIM4HBNzLNc5Uw0/',
    'QmzFdXt7enpyLY5PTs6n55dvjy/E8fX1uzdX9AdvFjqmlEk8aBr8/e6/29Q5j1FDraawwlf0VQJCJiqqrQtdQbFlhcpu/K+w',
    'D2/2yT47vDjlnUZCroiMecVq9QHU8MXRu6MCQthJ2tjT32L1PYyAhZpAfBvKRx5iP5dEy4iSltt2d6VlePNFQMRPCa0q7nM1',
    'gpYO/SrYesUIRlFdNyU2nyHO50AhraPx8wMyz8szZLaBThp1X1bQt02meGtpBW5chNOTM2wDmG68UseBy4tBGrfoC+KB9nMF',
    'smtjR3T6wDezti0W2IQAGqUinUF/o0ugVAh5WWhIwPfPQsW4dlnQhtcBVEaudhFlGusS9Q7lpfGMZMK7nfvx1n8AUEsBAhQA',
    'FAAAAAgAkTVDXesRLkI7BAAALgoAAC4AAAAAAAAAAAAAALaBAAAAAENvZGUvZXhwZXJpbWVudHMvY2hlY2tfazJfY292ZXJh',
    'Z2VfaWRlbnRpdHkucHlQSwECFAAUAAAACABRVUNdmQvpcWgDAACkBwAAKAAAAAAAAAAAAAAAtoGHBAAAQ29kZS9leHBlcmlt',
    'ZW50cy9jaGVja190aG0xMF9maXhlZF9rMi5weVBLAQIUABQAAAAIAMk2Q10HuqH0/wYAAKgNAAApAAAAAAAAAAAAAAC2gTUI',
    'AABNYXRoL3Byb29mcy9wcm9vZl9rMl9jb3ZlcmFnZV9pZGVudGl0eS5tZFBLAQIUABQAAAAIAMgzQ11f02OWdAcAAEgPAAAa',
    'AAAAAAAAAAAAAAC2gXsPAABNYXRoL3Byb29mcy9wcm9vZl90aG0wOS5tZFBLAQIUABQAAAAIABs3Q12viSx+rgkAAIEUAAAa',
    'AAAAAAAAAAAAAAC2gScXAABNYXRoL3Byb29mcy9wcm9vZl90aG0xMC5tZFBLAQIUABQAAAAIAEpUQ13MRcfQ0QYAAC8PAAAV',
    'AAAAAAAAAAAAAAC2gQ0hAABNYXRoL3RoZW9yZW1faW5kZXgubWRQSwECFAAUAAAACABKVENdKVy7BDgJAADlEgAAEwAAAAAA',
    'AAAAAAAAtoERKAAATWF0aC9wcm9vZl9hdWRpdC5tZFBLBQYAAAAABwAHAB0CAAB6MQAAAAA='
]))
(RUN_ROOT / "source_package.zip").write_bytes(SOURCE_ARCHIVE)
with zipfile.ZipFile(RUN_ROOT / "source_package.zip") as archive:
    for member in archive.infolist():
        target = (RUN_ROOT / member.filename).resolve()
        if not target.is_relative_to(RUN_ROOT.resolve()):
            raise RuntimeError(f"Unsafe ZIP member: {member.filename}")
    archive.extractall(RUN_ROOT)
record("sources", "passed", "THM-10 checks and proof, manuscript source and PDF")


## 3. Run K=2 coverage and THM-10 diagnostics


In [ ]:
for script, summary_relpath, expected_cases in [
    ("check_k2_coverage_identity.py", "fixed_k2_coverage/fixed_k2_coverage_summary.json", 5),
    ("check_thm10_fixed_k2.py", "fixed_k2_minimax/thm10_fixed_k2_summary.json", 10201),
]:
    if not RUN_OK:
        break
    started = time.perf_counter()
    command = [sys.executable, str(RUN_ROOT / "Code" / "experiments" / script)]
    result = subprocess.run(command, cwd=RUN_ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout[-5000:])
    if result.returncode:
        RUN_OK = False
        record(script, "failed", f"exit={result.returncode}")
    else:
        summary = json.loads((RUN_ROOT / "Code" / "results" / summary_relpath).read_text())
        count = summary.get("cases", summary.get("local_grid_checks"))
        RUN_OK = bool(summary["all_checks_passed"] and count == expected_cases)
        record(script, "passed" if RUN_OK else "failed", f"checks={count}; seconds={time.perf_counter()-started:.2f}")


## 4. Zip outputs and sources


In [ ]:
(RUN_ROOT / "notebook_stages.json").write_text(
    json.dumps({"run_ok": RUN_OK, "stages": STAGES}, indent=2), encoding="utf-8")
FINAL_ZIP = WORK_ROOT / "result_jcam_thm10_fixed_k2.zip"
with zipfile.ZipFile(FINAL_ZIP, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for path in sorted(RUN_ROOT.rglob("*")):
        if path.is_file() and path.name != "source_package.zip":
            archive.write(path, arcname=(Path("thm10_fixed_k2") / path.relative_to(RUN_ROOT)).as_posix())
print("Output archive:", FINAL_ZIP, "bytes:", FINAL_ZIP.stat().st_size, "run_ok:", RUN_OK)
